In [4]:
import cv2
import numpy as np
import pandas as pd
import os

In [5]:
# -------------------- IMAGE PATH -------------------- 
img_path = r"C:\Users\Admin\manali\colorpic.jpg"

In [6]:
# -------------------- VALIDATION -------------------- 
if not os.path.exists(img_path):
    print("❌ Error: Image path does not exist.") 
    exit()

In [7]:
# -------------------- LOAD IMAGE -------------------- 
img = cv2.imread(img_path) 
if img is None: 
    print("❌ Error: Failed to load image.") 
    exit()

In [8]:
# -------------------- GLOBAL VARIABLES -------------------- 
clicked = False 
r = g = b = xpos = ypos = 0

In [9]:
# -------------------- LOAD CSV -------------------- 
csv = pd.read_csv('colors.csv', names=["color","color_name","hex","R","G","B"], header=None)

In [10]:
# -------------------- COLOR FUNCTION --------------------
def getColorName(R, G, B):
    minimum = 10000
    cname = "Unknown"

    for i in range(len(csv)):
        d = abs(R - int(csv.loc[i, "R"])) + \
            abs(G - int(csv.loc[i, "G"])) + \
            abs(B - int(csv.loc[i, "B"]))

        if d <= minimum:
            minimum = d
            cname = csv.loc[i, "color_name"]

    return cname

In [11]:
# -------------------- MOUSE FUNCTION --------------------
def draw_function(event, x, y, flags, param):
    global b, g, r, xpos, ypos, clicked

    if event == cv2.EVENT_LBUTTONDBLCLK:
        clicked = True
        xpos = x
        ypos = y

        b, g, r = img[y, x]

        # ✅ FIX: convert to Python int
        b = int(b)
        g = int(g)
        r = int(r)

In [12]:
# -------------------- WINDOW --------------------
cv2.namedWindow('image')
cv2.setMouseCallback('image', draw_function)

# -------------------- LOOP --------------------
while True:
    img_copy = img.copy()  # prevent overwriting original

    if clicked:
        height, width, _ = img.shape

        # ✅ FIX: ensure color is valid tuple of ints
        color = (int(b), int(g), int(r))

        # Full-width bar
        cv2.rectangle(img_copy, (0, 0), (width, 70), color, -1)

        # Text
        text = getColorName(r, g, b) + f'  R={r} G={g} B={b}'

        # Text color based on brightness
        text_color = (0, 0, 0) if (r + g + b >= 600) else (255, 255, 255)

        cv2.putText(
            img_copy,
            text,
            (20, 45),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            text_color,
            2,
            cv2.LINE_AA
        )

    cv2.imshow("image", img_copy)

    # Press 'q' or ESC to exit
    key = cv2.waitKey(20) & 0xFF
    if key == ord('q') or key == 27:
        break

cv2.destroyAllWindows()